In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, countDistinct, avg, round as _round

In [0]:
SILVER = "delta/silver"
GOLD   = "delta/gold"

In [0]:
consolidated   = spark.table("silver.orders_consolidated")
reviews_bronze = spark.table("bronze.reviews")

In [0]:
avg_review_per_order = (
    reviews_bronze
    .groupBy("order_id")
    .agg(_round(avg("review_score"), 2).alias("avg_review_score"))
)

In [0]:
cons_with_review =  consolidated.join(avg_review_per_order, "order_id", "left")

In [0]:
customer_summary = (
    cons_with_review
    .groupBy("customer_id", "customer_city", "customer_state")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        _round(_sum(col("price") + col("freight_value")), 2).alias("total_revenue"),
        _round(_sum(col("price") + col("freight_value")) / countDistinct("order_id"), 2).alias("avg_order_value"),
        _round(avg("avg_review_score"), 2).alias("avg_review_score"),
    )
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

customer_summary.write.mode("overwrite").format("delta").saveAsTable("gold.customer_summary")
print(f"[GOLD] ✓ customer_summary | linhas: {customer_summary.count()}")

[GOLD] ✓ customer_summary | linhas: 97585


In [0]:
product_summary = (
    consolidated
    .groupBy("product_category_name")
    .agg(
        count("order_item_id").alias("total_units_sold"),
        _round(_sum(col("price") + col("freight_value")), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        _round(avg("freight_value"), 2).alias("avg_freight_value"),
    )
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

product_summary.write.mode("overwrite").format("delta").saveAsTable("gold.product_summary")
print(f"[GOLD] ✓ product_summary | linhas: {product_summary.count()}")

[GOLD] ✓ product_summary | linhas: 74


In [0]:
seller_summary = (
    cons_with_review
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        _round(_sum(col("price") + col("freight_value")), 2).alias("total_revenue"),
        _round(avg("avg_review_score"), 2).alias("avg_review_score"),
    )
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

seller_summary.write.mode("overwrite").format("delta").saveAsTable("gold.seller_summary")
print(f"[GOLD] ✓ seller_summary | linhas: {seller_summary.count()}")
print("[GOLD] Camada Gold concluída com sucesso.")

[GOLD] ✓ seller_summary | linhas: 2978
[GOLD] Camada Gold concluída com sucesso.


In [0]:
spark.stop()